In [1]:
import os
from langchain_openai import ChatOpenAI
from crewai_tools import MCPServerAdapter
from dotenv import load_dotenv

/home/ridwanfatur/work/portfolio/modular-ai/venvs/crewai-mcp-server/.venv/lib/python3.12/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/home/ridwanfatur/work/portfolio/modular-ai/venvs/crewai-mcp-server/.venv/lib/python3.12/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/ridwanfatur/work/portfolio/modular-ai/venvs/crewai-mcp-server/.venv/lib/python3.12/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/ridwanfatur/work/portfolio/modular-ai/venvs/crewai-mcp-server/.venv/lib/python3.12/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in

In [2]:
load_dotenv(override=True)

True

In [3]:
mcp_url = os.environ.get("MCP_SERVER_URL", "http://127.0.0.1:8150/sse")
mcp_url

'http://127.0.0.1:8150/sse'

In [4]:
mcp_adapter = MCPServerAdapter({"url": mcp_url, "transport": "sse"})

/home/ridwanfatur/work/portfolio/modular-ai/venvs/crewai-mcp-server/.venv/lib/python3.12/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'items', 'anyOf', 'enum', 'properties'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


In [5]:
len(mcp_adapter.tools)

8

In [6]:
def call_mcp_tool(adapter, tool_name, **kwargs):
    for tool in adapter.tools:
        if tool.name == tool_name:
            return tool.run(**kwargs)

    raise ValueError(f"Tool '{tool_name}' not found")

In [7]:
tables = call_mcp_tool(mcp_adapter, 'list_tables')
print(tables)

Using Tool: list_tables
table_name_89


In [8]:
result = call_mcp_tool(
    mcp_adapter,
    "run_sql_query",
    query="PRAGMA table_info(table_name_89);"
)
print(result)

Using Tool: run_sql_query
 cid    name type  notnull dflt_value  pk
   0   score TEXT        0       None   0
   1 visitor TEXT        0       None   0
   2  record TEXT        0       None   0


### Agent and Task

In [30]:
from crewai import Agent, Crew, Process, Task
from pydantic import BaseModel, Field

In [31]:
llm = ChatOpenAI(
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0,
    model_name=f"groq/llama-3.3-70b-versatile",
    top_p=1,
    max_retries=3,
    request_timeout=60,
)

allowed_tools = [
    "get_database_schema",
    "get_table_sample",
    "get_column_stats",
    "list_tables",
]
filtered_tools = [
    tool for tool in mcp_adapter.tools if tool.name in allowed_tools
]



class SQLQuery(BaseModel):
    """SQL query model for structured output from query generator"""
    sqlquery: str = Field(..., description="The raw sql query for the user input")

In [34]:
from crewai import Agent, Crew, Process, Task

agent_backstory = """
    "You are a SQL expert with 10+ years of experience. "
    "You ALWAYS start by examining the database schema carefully "
    "using get_database_schema tool to understand available tables "
    "and columns. When needed, you use get_table_sample and "
    "get_column_stats to understand data patterns. You can also "
    "use list_tables for a quick overview. "
    "You write clean, efficient SQL that strictly uses ONLY the "
    "tables and columns that exist in the schema. "
    "You NEVER invent table or column names. "
    "Your queries are always syntactically correct and executable. "
    "You output ONLY the SQL query without any markdown formatting "
    "or explanations."
"""

task_description = """
    MISSION: Generate a SQL query that answers this user question: "{user_input}"
    {previous_attempts_context}
    
    STEP-BY-STEP PROCESS:
    1. Use get_database_schema tool to see all available tables, columns, and sample data
    2. Identify which tables and columns are needed to answer the question
    3. If you need to understand data patterns, use get_table_sample or get_column_stats
    4. Write a SQL query using ONLY the tables and columns that exist in the schema
    5. Double-check that every table name and column name in your query exists in the schema
    
    CRITICAL RULES:
    - NEVER invent or assume table/column names - use get_database_schema first
    - Return ONLY the SQL query as plain text (no markdown, no ```sql, no explanations)
    - The query must be syntactically correct and executable
    - Use proper SQL syntax: JOINs, WHERE clauses, GROUP BY, ORDER BY, LIMIT as needed
    - If the question cannot be answered with available data, return:
      -- Cannot answer: [reason]
    
    Database schema is available via tools.
    USER QUESTION: {user_input}
"""

agent = Agent(
    role="Expert SQL Query Generator",
    goal=(
        "Convert natural language questions into precise, executable SQL "
        "queries that accurately answer the user's question"
    ),
    backstory=agent_backstory,
    max_iter=2,
    tools=filtered_tools,
    verbose=True,
    memory=False,
    llm=llm,
)

task = Task(
    description=task_description,
    expected_output=(
        "A single, clean SQL query without any formatting "
        "or explanation. Just the raw SQL text ready for execution."
    ),
    agent=agent,
    output_pydantic=SQLQuery,
)

crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    verbose=True
)

In [35]:
result = crew.kickoff(
    inputs={
        "user_input": "Name the score for toronto visitor and record of 29-17-8",
        "previous_attempts_context": ""
    }
)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fe691722-0e9a-401c-866e-4c966c15569e                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert SQL Query Generator                                                                              │
│                                                                                                                 │
│  Task:                                                                                                          │
│      MISSION: Generate a SQL query that answers this user question: "Name the score for toronto visitor and     │
│  record of 29-17-8"                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│      STEP-BY-STEP PROCESS:                                                                                      │
│      1. Use get_database_schema tool to see all available tables, columns, and sample data                      │
│      2. Identify which tables and columns are needed to answer the question                                     │
│      3. If you need to understand data patterns, use get_table_sample or get_column_stats                       │
│      4. Write a SQL query using ONLY the tables and columns that exist in the schema                            │
│      5. Double-check that every table name and column name in your query exists in the schema                   │
│                                                                                                                 │
│      CRITICAL RULES:                                                                                            │
│      - NEVER invent or assume table/column names - use get_database_schema first                                │
│      - Return ONLY the SQL query as plain text (no markdown, no ```sql, no explanations)                        │
│      - The query must be syntactically correct and executable                                                   │
│      - Use proper SQL syntax: JOINs, WHERE clauses, GROUP BY, ORDER BY, LIMIT as needed                         │
│      - If the question cannot be answered with available data, return:                                          │
│        -- Cannot answer: [reason]                                                                               │
│                                                                                                                 │
│      Database schema is available via tools.                                                                    │
│      USER QUESTION: Name the score for toronto visitor and record of 29-17-8                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/ridwanfatur/work/portfolio/modular-ai/venvs/crewai-mcp-server/.venv/lib/python3.12/site-packages/rich/live.py
:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert SQL Query Generator                                                                              │
│                                                                                                                 │
│  Thought: Thought: I need to understand the database schema to identify the relevant tables and columns to      │
│  answer the user's question.                                                                                    │
│                                                                                                                 │
│  Using Tool: get_database_schema                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{}"                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

 Maximum iterations reached. Requesting final answer.


/home/ridwanfatur/work/portfolio/modular-ai/venvs/crewai-mcp-server/.venv/lib/python3.12/site-packages/rich/live.py
:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert SQL Query Generator                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"sqlquery": "SELECT score FROM table_name_89 WHERE visitor = 'toronto' AND record = '29-17-8'"}               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 31e15ec5-332e-4000-8131-430f5d0abe87                                                                     │
│  Agent: Expert SQL Query Generator                                                                              │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: fe691722-0e9a-401c-866e-4c966c15569e                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {"sqlquery": "SELECT score FROM table_name_89 WHERE visitor = 'toronto' AND record =             │
│  '29-17-8'"}                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [37]:
result.pydantic

SQLQuery(sqlquery="SELECT score FROM table_name_89 WHERE visitor = 'toronto' AND record = '29-17-8'")